# Beyond Default Prediction: A Risk-Adjusted Approach to Loan Optimization

---

## Team Members

Point of Contact: Ashish Rajeev Chaudhary ( AshishRChaudhary )

Hazem Ani (hazemani), Kasem Chaisathitwanit (LuckyFriend), Prerna Bharti (PrernaBharti28)

## Overview

Lending institutions must decide which applicants are likely to repay their loans and which may default. These decisions determine both financial outcomes for the lender and access to credit for borrowers. In this project, we focus on a mid-size financial institution that wants to responsibly expand access to credit for borrowers with limited credit histories or lower credit scores while still protecting its financial stability.

Our objective is to build a model that estimates the probability that a borrower will default on a loan and use that information to evaluate different loan structures. Instead of treating lending as a simple approve-or-deny decision, we will examine how loan size affects the expected financial outcome of a loan. By estimating both the interest a lender may earn and the losses that could occur if a borrower defaults, we aim to calculate the expected financial value of different loan profiles. This allows us to explore how a lender might choose loan amounts that align with a specified level of risk tolerance.

The new aspect of our approach is that we move beyond default prediction alone. Most predictive models in lending focus only on identifying risky borrowers. Our approach integrates predicted default risk into a simple financial framework that estimates the expected dollar value of each loan profile. By simulating different loan sizes and repayment structures, we can evaluate how lending decisions change under different levels of risk tolerance.

If successful, this project will demonstrate how predictive modeling can support more flexible lending strategies. Rather than automatically denying applicants with weaker credit histories, lenders could identify loan structures that still produce positive expected value while managing risk. For the stakeholder, this provides a practical way to expand credit access to underserved borrowers while maintaining financial sustainability.

Our project objectives evolved from the initial proposal. The original plan focused primarily on predicting default risk, but we realized that prediction alone would not fully address the stakeholder’s decision problem or provide sufficient analytical depth for the project. As a result, we expanded the scope to include a financial decision framework that evaluates how loan size affects expected profitability under different levels of risk tolerance. This shift allows the analysis to produce more actionable recommendations rather than prediction alone.

## Data

#### Dataset Source<br>
The dataset used in this project comes from Kaggle.  
Dataset: Lending Club Loan Data  
https://www.kaggle.com/datasets/adarshsng/lending-club-loan-data-csv?select=loan.csv

**Data Creditability**<br>
The data originates from Lending Club, a peer-to-peer lending platform that provides public loan performance data for research and analysis. These datasets have been widely used in academic and industry research related to credit risk modeling.

The dataset contains around 2.26 million of loan records and over 140 variables describing things such as borrower credit history, loan characteristics, payment behavior, and credit account activity. Because the dataset reflects real financial transactions, it provides a realistic environment for building predictive models of default risk.

#### Exploratory Data Analysis
**Understanding the Dataset**<br>
Before modeling, we explored the dataset and reviewed the metadata documentation to understand the meaning of each variable. This helped us identify redundant or irrelevant features and determine which variables could potentially contribute to predicting loan default. 
We also performed exploratory analysis on numerical variables to examine their distributions and relationships.

**Distribution of Numerical Variables**<br>
Histograms were plotted for numerical variables to examine their distributions and detect potential outliers. One notable example was the annual income variable, which contained extremely large outliers. These outliers were removed to prevent them from disproportionately influencing the model.

**Correlation Analysis**<br>
We computed a correlation matrix for numerical variables and visualized it using a heatmap. This helped us identify highly correlated features that might introduce redundancy into the model. We found that loan_amnt and funded_amnt are high correlated. Because these variables contain very similar information, we removed funded_amnt to reduce redundancy.

![correlation_matrix](correlation_matrix.png)


## Preprocessing

Because the dataset is very large and contains a mixture of numeric and categorical variables, several preprocessing steps were necessary before training models.

**Feature Cleaning and Transformation**<br>
Several variables stored as strings were converted into numerical values. For example, the loan term column originally contained text values such as "36 months", "60 months", these were converted into integers.

**Handling Missing Values**<br>
First, Wwe examined the number of missing values in each column. For columns containing only a small number of missing observations (less than 2000), we removed rows with missing values to maintain data quality.

In [ ]:
small_null_cols = [col for col in loan_df.columns if 0 < loan_df[col].isnull().sum() <= 2000]
loan_df = loan_df.dropna(subset=small_null_cols)

For remaining variables, missing values were handled during model preparation using median imputation for numerical variables and mode imputation for categorical variables.

**Reducing High Cardinality Categorical Features**<br>
Some categorical variables contained extremely large numbers of unique values such as 'emp_title' and 'addr_state'. These variables were removed to reduce memory usage and simplify the model. These variables introduce high dimensionality when encoded and may not significantly improve predictive performance.

**Train-Test Split**<br>
The dataset was split into training and testing sets with train size of 70% and test size of 30%. Stratified sampling was used to maintain the class distribution of the target variable.

In [ ]:
from sklearn.model_selection import train_test_split

X = loan_df.drop(columns='loan_status')
y = loan_df['loan_status']

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.3,
                                                    random_state=42,
                                                    stratify=y
                                                    )

**Feature Encoding and Feature Scaling**
Categorical variables were encoded using One-Hot Encoding.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from scipy import sparse

encoder = OneHotEncoder(drop='first', sparse_output=True)

X_train_cat = encoder.fit_transform(X_train[cat_cols])
X_test_cat = encoder.transform(X_test[cat_cols])

Dropping the first category to avoid multicollinearity.

Numerical variables were scaled using StandardScaler. Because we were working with sparse matrices after encoding, we use StandardScaler(with_mean=False) to ensures compatibility with sparse data structures.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Only scale numerical columns, exclude target and binary columns
scaler = StandardScaler(with_mean=False)  # with_mean=False for sparse-compatible

X_train_num_scaled = scaler.fit_transform(X_train[num_cols])
X_test_num_scaled = scaler.transform(X_test[num_cols])

X_train_num_sparse = sparse.csr_matrix(X_train_num_scaled)
X_test_num_sparse = sparse.csr_matrix(X_test_num_scaled)

## Modeling

**Target Variable**<br>
The original loan_status variable contains multiple categories. For this project, we converted it into a binary classification problem.

**Baseline Model**<br>
Because the dataset is very large, training traditional logistic regression models became computationally expensive. Instead, we used Stochastic Gradient Descent (SGDClassifier) with logistic loss, which scales well to large datasets.  
The model was trained using class weights to address class imbalance: class_weight = {0:1, 1:4}

**Model Performance**<br>
Confusion Matrix:  
| Actual / Predicted | Fully Paid (0) | Default (1) |
|--------------------|---------------|-------------|
| Fully Paid (0)     | 257424       | 55178     |
| Default (1)        | 43889        | 34675      |

Classification Report:
| Class | Precision | Recall | F1-score | Support |
|------|-----------|--------|----------|---------|
| 0 | 0.85 | 0.82 | 0.84 | 312602 |
| 1 | 0.39 | 0.44 | 0.41 | 78564 |

Accuracy: 0.75

ROC-AUC Score: 0.71

The model achieves an overall accuracy of 0.75. The F1-score of 0.84 for class 0 shows strong performance in predicting this class. However, the performance for class 1 is weaker. The precision for class 1 is 0.39 and the recall is 0.44, with an F1-score of 0.41. This suggests that the model has difficulty correctly identifying class 1 cases and tends to misclassify many of them as class 0. Overall, the model performs reasonably well for the majority class but struggles with the minority class.

## Problems and Challenges

There were several challenges arose during the project.

**Large Dataset Size**<br>
The dataset contains millions of records and many features, making memory management and model training computationally intensive. To tackle this problem, we converted categorical features to sparse matrices and used scalable algorithms such as SGDClassifier which can handle large dataset better than LogisticRegression.

**High Dimensionality**<br>
Some categorical variables contained extremely large numbers of unique values. When encoded, these variables dramatically increased feature dimensionality. We mitigated this by removing columns with high number of unique values.

**Class Imbalance**<br>
The dataset contains more fully paid loans than defaulted loans. This imbalance can bias the model toward predicting the majority class. To address this problem, we used class weighting during model training.


## Next Steps


**Advanced Models (M3.T2)**
- Train Random Forest or XGBoost expected ROC-AUC to go up
- Compare performance against logistic regression baseline

**Model Evaluation & Selection (M3.T3-M3.T5)**
- Cross-validation on all candidate models
- Evaluate using ROC-AUC, Precision-Recall, and F1-score
- Calibrate predicted probabilities using Platt scaling

**Financial Model Construction (M4)**
- Build a separate regression model to estimate expected profit per loan.

**Loan Size Optimization (M5)**
- Build regression model to simulate different loan amounts per borrower.
- Identify profit maximizing loan size under lender's risk tolerance.